# 1. Introducción
Notebook base de análisis exploratorio para fotonIA.

# 2. Carga de datos

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'dataset.csv'
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset_clean.csv'
METRICS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'eda_metrics.json'

def build_sample_data(rows: int = 120) -> pd.DataFrame:
    rng = np.random.default_rng(42)
    regions = ['North', 'South', 'East', 'West']
    months = [f'2026-{m:02d}' for m in range(1, 13)]

    df_sample = pd.DataFrame({
        'region': rng.choice(regions, size=rows),
        'month': rng.choice(months, size=rows),
        'generation_cost': rng.normal(44, 4, size=rows).round(2),
        'distribution_cost': rng.normal(17, 2, size=rows).round(2),
        'commercialization_cost': rng.normal(8, 1, size=rows).round(2),
        'energy_consumption_kwh': rng.normal(1150, 180, size=rows).round(0)
    })
    df_sample['unit_cost'] = (
        df_sample['generation_cost'] + df_sample['distribution_cost'] + df_sample['commercialization_cost']
    ).round(2)

    missing_idx = rng.choice(df_sample.index, size=max(1, rows // 20), replace=False)
    df_sample.loc[missing_idx, 'distribution_cost'] = np.nan

    return df_sample

if RAW_PATH.exists():
    df = pd.read_csv(RAW_PATH)
    print(f'Loaded real dataset: {RAW_PATH}')
else:
    df = build_sample_data()
    print('Raw dataset not found. Using generated sample data.')

df.head()

# 3. Exploración inicial

In [ ]:
print('Shape:', df.shape)
display(df.info())
display(df.describe(include='all').T)

# 4. Normalización de nombres de columnas

In [ ]:
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('-', '_', regex=False)
)
df.columns.tolist()

# 5. Evaluación de calidad de datos

In [ ]:
quality_report = {
    'rows': int(df.shape[0]),
    'columns': int(df.shape[1]),
    'missing_values_total': int(df.isna().sum().sum()),
    'missing_values_by_column': df.isna().sum().astype(int).to_dict(),
    'duplicated_rows': int(df.duplicated().sum()),
    'data_types': {c: str(t) for c, t in df.dtypes.items()}
}
quality_report

# 6. Tratamiento de datos faltantes

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

categorical_cols = df.select_dtypes(exclude=[np.number]).columns
for col in categorical_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].mode().iloc[0])

df.isna().sum()

# 7. Eliminación de duplicados

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)
print('Removed duplicates:', before - after)

# 8. Análisis univariado

In [ ]:
display(df.describe().T)
df['region'].value_counts()

# 9. Análisis bivariado

In [ ]:
region_cost = df.groupby('region', as_index=False)['unit_cost'].mean()
display(region_cost)
sns.barplot(data=region_cost, x='region', y='unit_cost')
plt.title('Average Unit Cost by Region')
plt.show()

# 10. Análisis multivariado

In [ ]:
corr = df.select_dtypes(include=[np.number]).corr(numeric_only=True)
plt.figure(figsize=(8, 5))
sns.heatmap(corr, annot=True, cmap='YlGnBu', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

# 11. Detección de outliers con IQR

In [ ]:
outlier_rows = []
for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    count = int(((df[col] < low) | (df[col] > high)).sum())
    outlier_rows.append({'variable': col, 'low': float(low), 'high': float(high), 'outlier_count': count})

outliers_df = pd.DataFrame(outlier_rows)
outliers_df

# 12. Visualización de resultados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df['unit_cost'], kde=True, ax=axes[0], color='teal')
axes[0].set_title('Unit Cost Distribution')

monthly = df.groupby('month', as_index=False)['energy_consumption_kwh'].mean().sort_values('month')
sns.lineplot(data=monthly, x='month', y='energy_consumption_kwh', marker='o', ax=axes[1], color='darkorange')
axes[1].set_title('Average Consumption by Month')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# 13. Exportación de dataset limpio a data/processed/dataset_clean.csv

In [ ]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)
print(f'Clean dataset exported to: {PROCESSED_PATH}')

# 14. Exportación opcional de métricas a JSON

In [ ]:
metrics = {
    'summary': {
        'rows': int(df.shape[0]),
        'columns': int(df.shape[1]),
        'missing_values_total': int(df.isna().sum().sum()),
        'duplicated_rows': int(df.duplicated().sum())
    },
    'descriptive_stats': df.describe(include='all').fillna('').to_dict(),
    'outliers': outliers_df.to_dict(orient='records')
}

with open(METRICS_PATH, 'w', encoding='utf-8') as fp:
    json.dump(metrics, fp, indent=2)

print(f'Metrics exported to: {METRICS_PATH}')

# 15. Conclusiones preliminares
- Se observan diferencias de costo unitario por región.
- Hay variaciones temporales en el consumo energético.
- El flujo permite evolucionar desde exploración hasta una base para modelos futuros.